# Advanced Account Enrichment Pipeline

Build a production-grade enrichment pipeline with **multi-step execution**, **conditional logic**, **caching**, **hooks**, and **provider flexibility**.

**What you'll learn:**
- Chain **FunctionStep → LLMStep** with automatic DAG resolution
- Use **`run_if`** to skip expensive steps on low-value rows
- Enable **SQLite caching** for zero-cost reruns
- Wire **lifecycle hooks** for observability
- Swap **providers** (OpenAI, Anthropic, Google) and enable **grounding**

> **Prerequisites:** Complete [quickstart.ipynb](quickstart.ipynb) first.

## 1. Setup & Configuration

In [1]:
import pandas as pd

from accrue import EnrichmentConfig, EnrichmentHooks, FunctionStep, LLMStep, Pipeline

pd.set_option("display.max_colwidth", 80)

companies = [
    {
        "company": "Stripe",
        "website": "stripe.com",
        "description": "Online payment processing for internet businesses",
    },
    {
        "company": "Notion",
        "website": "notion.so",
        "description": "All-in-one workspace for notes, docs, and project management",
    },
    {
        "company": "Figma",
        "website": "figma.com",
        "description": "Collaborative interface design tool for teams",
    },
    {
        "company": "Datadog",
        "website": "datadoghq.com",
        "description": "Cloud monitoring and security platform",
    },
    {
        "company": "Canva",
        "website": "canva.com",
        "description": "Online design and visual communication platform",
    },
    {
        "company": "Plaid",
        "website": "plaid.com",
        "description": "Financial data connectivity for fintech applications",
    },
    {
        "company": "Airtable",
        "website": "airtable.com",
        "description": "Low-code platform for building collaborative apps",
    },
    {
        "company": "Vercel",
        "website": "vercel.com",
        "description": "Frontend cloud platform for web developers",
    },
]

/Users/matthew.house/Programming/lattice/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# EnrichmentConfig presets control concurrency, caching, logging, and more.
# Choose the preset that matches your environment:

dev_config = EnrichmentConfig.for_development()  # 5 workers, debug logging, caching on
prod_config = EnrichmentConfig.for_production()  # 30 workers, checkpointing, caching
server_config = EnrichmentConfig.for_server()  # No progress bar, high concurrency, warnings only

# We'll use the development preset for this notebook
config = EnrichmentConfig.for_development()
print(
    f"Workers: {config.max_workers}, Caching: {config.enable_caching}, Log level: {config.log_level}"
)

Workers: 5, Caching: True, Log level: DEBUG


---

## 2. Multi-Step Account Enrichment

A realistic pipeline: **FunctionStep** for data lookup, then two independent **LLMSteps** that run in parallel (no dependency between them — Accrue's DAG resolver handles this automatically).

In [3]:
def lookup_account_data(ctx):
    """Simulate CRM/enrichment API (Clearbit, ZoomInfo, Apollo, etc.).
    In production, replace with real API calls."""
    database = {
        "Stripe": {
            "hq": "San Francisco, CA",
            "founded": 2010,
            "funding": "$2.3B",
            "employees": 8000,
            "tech_stack": ["AWS", "Ruby", "Go"],
        },
        "Notion": {
            "hq": "San Francisco, CA",
            "founded": 2013,
            "funding": "$343M",
            "employees": 800,
            "tech_stack": ["AWS", "React", "Kotlin"],
        },
        "Figma": {
            "hq": "San Francisco, CA",
            "founded": 2012,
            "funding": "$333M",
            "employees": 1500,
            "tech_stack": ["AWS", "C++", "TypeScript"],
        },
        "Datadog": {
            "hq": "New York, NY",
            "founded": 2010,
            "funding": "$648M",
            "employees": 5000,
            "tech_stack": ["AWS", "Go", "Python"],
        },
        "Canva": {
            "hq": "Sydney, Australia",
            "founded": 2012,
            "funding": "$572M",
            "employees": 4000,
            "tech_stack": ["AWS", "Java", "TypeScript"],
        },
        "Plaid": {
            "hq": "San Francisco, CA",
            "founded": 2013,
            "funding": "$734M",
            "employees": 1000,
            "tech_stack": ["AWS", "Python", "Go"],
        },
        "Airtable": {
            "hq": "San Francisco, CA",
            "founded": 2012,
            "funding": "$1.4B",
            "employees": 900,
            "tech_stack": ["AWS", "Node.js", "React"],
        },
        "Vercel": {
            "hq": "San Francisco, CA",
            "founded": 2015,
            "funding": "$563M",
            "employees": 500,
            "tech_stack": ["AWS", "Go", "TypeScript"],
        },
    }
    company = ctx.row["company"]
    data = database.get(company, {})
    return {"__account_data": data}


pipeline = Pipeline(
    [
        FunctionStep("lookup", fn=lookup_account_data, fields=["__account_data"]),
        # These two steps both depend on "lookup" but NOT on each other —
        # Accrue runs them in parallel automatically.
        LLMStep(
            "classify",
            fields={
                "segment": {
                    "prompt": "Classify into market segment based on the account data",
                    "enum": ["Enterprise", "Mid-Market", "SMB"],
                },
                "icp_score": {
                    "prompt": "Score 1-100 for fit as a B2B SaaS prospect. Consider: employee count, funding stage, tech stack sophistication",
                    "type": "Number",
                    "default": 0,
                },
            },
            depends_on=["lookup"],
        ),
        LLMStep(
            "competitive_intel",
            fields={
                "competitive_position": {
                    "prompt": "Rate competitive position in their market",
                    "enum": ["Market Leader", "Challenger", "Niche Player"],
                },
                "key_competitors": {
                    "prompt": "List top 3 competitors",
                    "type": "List[String]",
                },
            },
            depends_on=["lookup"],
        ),
    ]
)

result = await pipeline.run_async(companies, config=config)
pd.DataFrame(result.data)[
    ["company", "segment", "icp_score", "competitive_position", "key_competitors"]
]

Pipeline:  33%|███▎      | 1/3 [00:00<00:00, 205.17step/s, step=classify, competitive_intel]








Pipeline: 100%|██████████| 3/3 [00:05<00:00,  1.70s/step, step=classify, competitive_intel] 


,company,segment,icp_score,competitive_position,key_competitors
0,Stripe,Enterprise,95.0,Market Leader,"[PayPal, Square, Adyen]"
1,Notion,Mid-Market,85.0,Market Leader,"[Evernote, Microsoft OneNote, Coda]"
2,Figma,Enterprise,90.0,Market Leader,"[Adobe XD, Sketch, InVision]"
3,Datadog,Enterprise,95.0,Market Leader,"[New Relic, Splunk, Dynatrace]"
4,Canva,Enterprise,90.0,Market Leader,"[Adobe Spark, Figma, Piktochart]"
5,Plaid,Enterprise,90.0,Market Leader,"[Plaid, Yodlee, Finicity]"
6,Airtable,Mid-Market,85.0,Challenger,"[Smartsheet, Monday.com, Notion]"
7,Vercel,Mid-Market,85.0,Market Leader,"[Netlify, GitHub Pages, Firebase Hosting]"


> **Note:** `__account_data` doesn't appear in the output — fields prefixed with `__` are **internal** (passed between steps, filtered from final results).

---

## 3. Conditional Steps — Cost Optimization

Only do expensive deep analysis on **high-ICP accounts**. The `run_if` predicate checks `prior_results` from upstream steps — if the account scored below 70, the expensive step is **skipped entirely**.

In [4]:
pipeline = Pipeline(
    [
        FunctionStep("lookup", fn=lookup_account_data, fields=["__account_data"]),
        LLMStep(
            "qualify",
            fields={
                "icp_score": {
                    "prompt": "Score 1-100 for ICP fit as a B2B SaaS prospect",
                    "type": "Number",
                    "default": 0,
                },
                "segment": {
                    "prompt": "Classify market segment",
                    "enum": ["Enterprise", "Mid-Market", "SMB"],
                },
            },
            depends_on=["lookup"],
        ),
        # Only runs for accounts with icp_score >= 70
        LLMStep(
            "deep_analysis",
            fields={
                "executive_summary": "Write a 3-sentence executive briefing for the sales team",
                "recommended_approach": {
                    "prompt": "Suggest the best sales approach",
                    "enum": [
                        "Product-Led",
                        "Enterprise Sales",
                        "Partner Channel",
                        "Self-Serve",
                    ],
                },
            },
            depends_on=["qualify"],
            run_if=lambda row, prior: prior.get("icp_score", 0) >= 70,
        ),
    ]
)

result = await pipeline.run_async(companies, config=config)

# Check how many rows were skipped
deep_usage = result.cost.steps.get("deep_analysis")
if deep_usage:
    print(
        f"Deep analysis: {deep_usage.rows_processed} processed, {deep_usage.rows_skipped} skipped"
    )
    print(
        f"Cost saved by skipping: ~{deep_usage.rows_skipped / (deep_usage.rows_processed + deep_usage.rows_skipped):.0%}"
    )

pd.DataFrame(result.data)[
    ["company", "icp_score", "segment", "executive_summary", "recommended_approach"]
]

Pipeline: 100%|██████████| 3/3 [00:08<00:00,  2.84s/step, step=deep_analysis]

Deep analysis: 8 processed, 0 skipped
Cost saved by skipping: ~0%


,company,icp_score,segment,executive_summary,recommended_approach
0,Stripe,85.0,Enterprise,Stripe is a leading online payment processing platform tailored for internet...,Enterprise Sales
1,Notion,85.0,Mid-Market,"Notion is a versatile all-in-one workspace that integrates notes, documents,...",Product-Led
2,Figma,85.0,Mid-Market,"Figma is a collaborative interface design tool tailored for teams, emphasizi...",Product-Led
3,Datadog,90.0,Enterprise,Datadog is a leading cloud monitoring and security platform catering primari...,Enterprise Sales
4,Canva,85.0,Enterprise,Canva is a leading online design and visual communication platform with a st...,Enterprise Sales
5,Plaid,85.0,Mid-Market,"Plaid specializes in financial data connectivity, enabling fintech applicati...",Product-Led
6,Airtable,85.0,Mid-Market,Airtable is a low-code platform that enables users to build collaborative ap...,Product-Led
7,Vercel,85.0,Mid-Market,"Vercel is a leading frontend cloud platform tailored for web developers, ena...",Product-Led


Accounts below 70 ICP score get `None` for `executive_summary` and `recommended_approach` — you only paid for the accounts that matter.

---

## 4. Caching — Never Pay Twice

Enable caching and run the same pipeline twice. The second run serves everything from **SQLite cache** — zero API calls, zero cost.

In [5]:
config = EnrichmentConfig(enable_caching=True)

pipeline = Pipeline(
    [
        LLMStep(
            "enrich",
            fields={
                "industry": {
                    "prompt": "Classify the company's primary industry",
                    "enum": [
                        "Fintech",
                        "Developer Tools",
                        "Productivity",
                        "Design",
                        "Infrastructure",
                        "Other",
                    ],
                },
            },
        )
    ]
)

# First run — calls the API
result1 = await pipeline.run_async(companies, config=config)
usage1 = result1.cost.steps["enrich"]
print(
    f"Run 1: {usage1.cache_hits} cache hits, {usage1.cache_misses} misses, {result1.cost.total_tokens:,} tokens"
)

# Second run — everything from cache
result2 = await pipeline.run_async(companies, config=config)
usage2 = result2.cost.steps["enrich"]
print(
    f"Run 2: {usage2.cache_hits} cache hits, {usage2.cache_misses} misses, {result2.cost.total_tokens:,} tokens"
)

Pipeline: 100%|██████████| 1/1 [00:01<00:00,  1.67s/step, step=enrich]


Run 1: 0 cache hits, 8 misses, 3,127 tokens


Pipeline: 100%|██████████| 1/1 [00:00<00:00, 103.21step/s, step=enrich]

Run 2: 8 cache hits, 0 misses, 0 tokens


In [6]:
# Clear cache when you want a fresh start
deleted = pipeline.clear_cache()
print(f"Cleared {deleted} cached entries")

Cleared 48 cached entries


> **Note:** Cache keys include **step name**, **row data**, **field specs**, **model**, and **temperature**. Change a prompt and only that step re-runs — everything else serves from cache.

---

## 5. Lifecycle Hooks — Observability

Wire up callbacks for **real-time progress tracking** and **cost monitoring**. Hook errors are caught and logged — they never crash your pipeline.

In [7]:
from accrue import PipelineEndEvent, RowCompleteEvent, StepEndEvent


def on_row_complete(event: RowCompleteEvent):
    """Track progress as each account completes."""
    status = "cached" if event.from_cache else "skipped" if event.skipped else "enriched"
    print(f"  [{event.step_name}] Row {event.row_index + 1}: {status}")


def on_step_end(event: StepEndEvent):
    """Monitor cost per step."""
    tokens = event.usage.total_tokens if event.usage else 0
    print(f"Step '{event.step_name}' done: {tokens:,} tokens, {event.elapsed_seconds:.1f}s")


def on_pipeline_end(event: PipelineEndEvent):
    """Summary when pipeline finishes."""
    print(
        f"\nPipeline complete: {event.num_rows} rows, {event.cost.total_tokens:,} tokens, {event.elapsed_seconds:.1f}s"
    )


hooks = EnrichmentHooks(
    on_row_complete=on_row_complete,
    on_step_end=on_step_end,
    on_pipeline_end=on_pipeline_end,
)

pipeline = Pipeline(
    [
        LLMStep(
            "enrich",
            fields={
                "industry": {
                    "prompt": "Classify the company's primary industry",
                    "enum": [
                        "Fintech",
                        "Developer Tools",
                        "Productivity",
                        "Design",
                        "Infrastructure",
                        "Other",
                    ],
                },
            },
        )
    ]
)

result = await pipeline.run_async(companies, hooks=hooks)

Pipeline:   0%|          | 0/1 [00:00<?, ?step/s, step=enrich]

  [enrich] Row 6: enriched
  [enrich] Row 5: enriched
  [enrich] Row 1: enriched
  [enrich] Row 7: enriched
  [enrich] Row 2: enriched


Pipeline: 100%|██████████| 1/1 [00:00<00:00,  1.07step/s, step=enrich]

  [enrich] Row 3: enriched
  [enrich] Row 8: enriched
  [enrich] Row 4: enriched
Step 'enrich' done: 3,127 tokens, 0.9s

Pipeline complete: 8 rows, 3,127 tokens, 0.9s


---

## 6. Provider Flexibility & Grounding

### Multi-provider

Swap providers with one line. Same pipeline, different LLM.

In [8]:
# OpenAI — default, zero config
LLMStep("enrich", fields={"industry": "Classify the company's primary industry"})

# Anthropic (pip install accrue[anthropic])
from accrue.providers import AnthropicClient

LLMStep(
    "enrich",
    fields={"industry": "Classify the company's primary industry"},
    model="claude-sonnet-4-5-20250929",
    client=AnthropicClient(),
)

# Google (pip install accrue[google])
from accrue.providers import GoogleClient

LLMStep(
    "enrich",
    fields={"industry": "Classify the company's primary industry"},
    model="gemini-2.5-flash",
    client=GoogleClient(),
)

# OpenAI-compatible (Ollama, Groq, Together, etc.)
LLMStep(
    "enrich",
    fields={"industry": "Classify the company's primary industry"},
    model="llama3",
    base_url="http://localhost:11434/v1",
)

print("Provider examples created (not executed — set API keys and uncomment to run)")

Provider examples created (not executed — set API keys and uncomment to run)


### Grounding — Real-Time Web Data

**Grounding** lets the LLM search the web natively — no separate FunctionStep needed. One step, real-time data.

In [9]:
# Basic grounding — LLM searches the web before answering
grounded_pipeline = Pipeline(
    [
        LLMStep(
            "research",
            fields={
                "recent_news": "Summarize the most significant recent news about this company",
                "market_position": "Describe their current market position and trajectory",
            },
            grounding=True,
        )
    ]
)

# Uncomment to run (requires OpenAI API key, uses web search tokens):
# result = await grounded_pipeline.run_async(companies[:2])
# pd.DataFrame(result.data)[["company", "recent_news", "market_position", "sources"]]

In [10]:
# Grounding with domain filtering
filtered_pipeline = Pipeline(
    [
        LLMStep(
            "research",
            fields={
                "funding_history": "Summarize the company's funding history and latest round",
            },
            grounding={
                "allowed_domains": ["crunchbase.com", "techcrunch.com", "sec.gov"],
                "blocked_domains": ["reddit.com"],
            },
        )
    ]
)

# Grounding works with any provider
LLMStep(
    "research",
    fields={...},
    grounding=True,
    client=AnthropicClient(),
    model="claude-sonnet-4-5-20250929",
)

print("Grounding examples created (uncomment to run with real API calls)")

Grounding examples created (uncomment to run with real API calls)


---

## 7. Putting It All Together

A full production pipeline combining everything: **data lookup**, **classification** with **conditional deep analysis**, **caching**, and **hooks**.

In [11]:
# --- Full production pipeline ---


def on_complete(event: PipelineEndEvent):
    print(
        f"Done: {event.num_rows} rows, {event.cost.total_tokens:,} tokens, {event.elapsed_seconds:.1f}s"
    )
    if event.total_errors:
        print(f"  Errors: {event.total_errors}")


pipeline = Pipeline(
    [
        # Step 1: Data lookup (FunctionStep — your APIs, databases, CRM)
        FunctionStep("lookup", fn=lookup_account_data, fields=["__account_data"]),
        # Step 2: Classify and score (LLM)
        LLMStep(
            "classify",
            fields={
                "segment": {
                    "prompt": "Classify into market segment based on the account data",
                    "enum": ["Enterprise", "Mid-Market", "SMB"],
                },
                "icp_score": {
                    "prompt": "Score 1-100 for fit as a B2B SaaS prospect",
                    "type": "Number",
                    "default": 0,
                },
            },
            depends_on=["lookup"],
        ),
        # Step 3: Deep analysis — only for high-ICP accounts (conditional)
        LLMStep(
            "deep_analysis",
            fields={
                "executive_summary": "Write a 3-sentence executive briefing for the sales team",
                "recommended_approach": {
                    "prompt": "Suggest the best sales approach",
                    "enum": [
                        "Product-Led",
                        "Enterprise Sales",
                        "Partner Channel",
                        "Self-Serve",
                    ],
                },
            },
            depends_on=["classify"],
            run_if=lambda row, prior: prior.get("icp_score", 0) >= 70,
        ),
    ]
)

config = EnrichmentConfig(
    enable_caching=True,
    max_workers=10,
)

hooks = EnrichmentHooks(on_pipeline_end=on_complete)

result = await pipeline.run_async(companies, config=config, hooks=hooks)
pd.DataFrame(result.data)[
    ["company", "segment", "icp_score", "executive_summary", "recommended_approach"]
]

Pipeline: 100%|██████████| 3/3 [00:04<00:00,  1.45s/step, step=deep_analysis]

Done: 8 rows, 9,145 tokens, 4.3s


,company,segment,icp_score,executive_summary,recommended_approach
0,Stripe,Enterprise,90.0,Stripe is a leading provider of online payment processing solutions tailored...,Enterprise Sales
1,Notion,Mid-Market,85.0,"Notion is a versatile all-in-one workspace that integrates notes, documents,...",Product-Led
2,Figma,Enterprise,85.0,"Figma is a collaborative interface design tool tailored for teams, emphasizi...",Enterprise Sales
3,Datadog,Enterprise,90.0,Datadog is a leading cloud monitoring and security platform catering primari...,Enterprise Sales
4,Canva,Enterprise,85.0,Canva is a leading online design and visual communication platform that serv...,Product-Led
5,Plaid,Enterprise,85.0,Plaid is a leading provider of financial data connectivity solutions tailore...,Enterprise Sales
6,Airtable,Mid-Market,85.0,Airtable is a leading low-code platform that enables users to build collabor...,Product-Led
7,Vercel,Mid-Market,85.0,"Vercel is a leading frontend cloud platform tailored for web developers, ena...",Product-Led


In [12]:
# Run again — everything from cache
result2 = await pipeline.run_async(companies, config=config, hooks=hooks)
print(f"\nCache hit rate: {result2.cost.steps['classify'].cache_hit_rate:.0%}")

Pipeline: 100%|██████████| 3/3 [00:00<00:00, 223.31step/s, step=deep_analysis]

Done: 8 rows, 0 tokens, 0.0s

Cache hit rate: 100%


In [13]:
# Clean up
pipeline.clear_cache()

24